# 24.4 设计广告 CTR 系统 / Design an Ads CTR System (Meta / Google Ads)

**中文**:在线广告是互联网最赚钱的生意(Google、Meta 收入的大头),其核心是一道每秒发生百万次的**竞价排序**问题。它和推荐(24.1)共享排序漏斗,但有一个**决定性的独特要求——校准(calibration)**。因为广告排序不是简单按"最可能被点"排,而是按 **eCPM = 出价(bid)× 预测点击率(pCTR)** 排(谁给平台带来的期望收入高谁排前面)。这意味着 **pCTR 必须是"校准过"的真实概率**——预测 2% 就得真的有 2% 的人点,而不只是"排序对就行"。一个 AUC 很高(排序很准)但**系统性高估** pCTR 的模型,会让整个竞价体系算错出价、排错广告、让广告主超支或平台亏损。本节从零演示"排序好 ≠ 概率准",以及校准如何在**不改变排序能力**的前提下修正概率,再讲清广告系统设计的完整框架。
**English**: Online advertising is the internet's most profitable business (the bulk of Google's and Meta's revenue), centered on an **auction-ranking** problem happening millions of times per second. It shares the ranking funnel with recommendation (24.1) but has a **decisive unique requirement — calibration**. Because ad ranking isn't simply "most likely to be clicked" but by **eCPM = bid × predicted CTR (pCTR)** (whoever brings the platform higher expected revenue ranks first). This means **pCTR must be a "calibrated" true probability** — a predicted 2% must actually be clicked by 2% of people, not just "ranked correctly." A model with high AUC (accurate ranking) that **systematically overestimates** pCTR will make the whole auction miscompute bids, mis-rank ads, and cause advertisers to overspend or the platform to lose money. This section demonstrates "good ranking ≠ accurate probability" from scratch, and how calibration fixes probabilities **without changing ranking ability**, then clarifies the complete ads-system-design framework.

---

**中文**:**广告竞价的核心机制**:每次展示广告,平台在众多候选广告里跑一个**竞价(auction)**:
**English**: **The core mechanism of the ad auction**: for each ad impression, the platform runs an **auction** among many candidate ads:
$$\text{eCPM}=\text{bid}\times \text{pCTR}\times 1000 \quad(\text{每千次展示的期望收入})$$
**中文**:按 eCPM 排序,最高者胜出展示(常用 **GSP 广义第二价格**:胜者付第二名的价格)。**关键**:排序依赖 **pCTR**,而 pCTR 是模型预测的——如果预测不准(尤其系统性偏差),整个竞价就错了。
**English**: Rank by eCPM, and the highest wins the impression (often **GSP generalized second price**: the winner pays the second-place price). **Key**: ranking depends on **pCTR**, which the model predicts — if the prediction is inaccurate (especially systematically biased), the whole auction is wrong.

**中文**:**为什么"校准"对广告是生死攸关的(而对推荐相对次要)**:
**English**: **Why "calibration" is life-or-death for ads (but relatively secondary for recommendation)**:
- **中文**:**推荐**:只需**排序对**(把用户最可能喜欢的排前面),pCTR 是 0.3 还是 0.6 无所谓,只要相对顺序对——所以推荐可以只优化排序(AUC/NDCG)。
  **Recommendation**: only needs **correct ranking** (most-liked items first); whether pCTR is 0.3 or 0.6 doesn't matter as long as relative order is right — so recommendation can optimize only ranking (AUC/NDCG).
- **中文**:**广告**:eCPM = bid × pCTR,pCTR 的**绝对值**直接进入出价计算。如果模型把所有 pCTR 系统性高估 2 倍,eCPM 全错、竞价排序错、计费错(广告主为虚高的预期点击付钱)。所以广告**必须校准**——预测概率要等于真实频率。这是广告 CTR 模型区别于普通排序模型的本质。
  **Ads**: eCPM = bid × pCTR, and pCTR's **absolute value** directly enters the bid computation. If the model systematically overestimates all pCTRs by 2x, eCPM is all wrong, auction ranking is wrong, billing is wrong (advertisers pay for inflated expected clicks). So ads **must be calibrated** — predicted probability must equal true frequency. This is the essence distinguishing an ads-CTR model from an ordinary ranking model.

> 💡 **面试速查 / Interview cheat-sheet（★★★ 广告系统设计, 高频）**
> **中文**:**广告 CTR 系统**:漏斗(候选广告→CTR/CVR 预估→**竞价排序**), 核心是 **eCPM=bid×pCTR** 排序(GSP 二价计费)。**决定性要求=校准**:pCTR 必须是真实概率(预测2%就真2%点), 因为绝对值直接进出价——**排序好(高AUC)≠概率准**, 系统性偏差会让竞价/计费全错。校准方法:**Platt scaling / 保序回归(isotonic)** 在留出集上把分数映射到真实频率; 指标 **ECE/log loss/校准曲线**(AUC 不衡量校准!)。**模型**:大规模稀疏特征(用户×广告×上下文交叉), LR→**FM/FFM→Wide&Deep/DeepFM/DCN**(记忆+泛化), 特征哈希、embedding。**关键难题**:①**极端规模**(十亿次/天预测, 毫秒延迟)②**类别极不平衡**(CTR 常 <1%)③**反馈回路/选择偏差**(只观测到展示过的广告的点击→未展示的没标签, 需探索/无偏)④**实时性**(在线学习/增量更新追时效)⑤**多目标**(CTR×CVR×出价, 转化漏斗 ESMM)⑥预算平滑(budget pacing)。**评估**:离线 AUC+**校准(ECE/logloss)** + 在线 A/B(收入、ROI、广告主满意度、用户体验护栏)。面试金句:*"广告 CTR 系统按 eCPM=bid×pCTR 竞价排序, 所以 pCTR 必须校准(真实概率), 排序好≠概率准, 高AUC模型若系统性高估会毁掉竞价计费——用 Platt/保序回归校准、看 ECE 而非只看 AUC; 挑战是十亿级规模、类别极不平衡、只观测展示广告的选择偏差(需探索)、实时性; 特征稀疏交叉用 DeepFM/DCN, 多目标 CTR×CVR 用 ESMM。"*
> **English**: **Ads CTR system**: funnel (candidate ads → CTR/CVR estimation → **auction ranking**), core is **eCPM = bid × pCTR** ranking (GSP second-price billing). **Decisive requirement = calibration**: pCTR must be a true probability (predicted 2% actually clicked 2%), because its absolute value directly enters the bid — **good ranking (high AUC) ≠ accurate probability**, and systematic bias wrecks auction/billing. Calibration methods: **Platt scaling / isotonic regression** mapping scores to true frequency on a holdout; metrics **ECE/log loss/calibration curve** (AUC doesn't measure calibration!). **Models**: large-scale sparse features (user × ad × context crosses), LR → **FM/FFM → Wide&Deep/DeepFM/DCN** (memorization + generalization), feature hashing, embeddings. **Key challenges**: ① **extreme scale** (billions/day predictions, ms latency) ② **extreme class imbalance** (CTR often <1%) ③ **feedback loops/selection bias** (only observe clicks on shown ads → unshown have no labels, needs exploration/unbiased) ④ **freshness** (online learning/incremental updates for timeliness) ⑤ **multi-objective** (CTR × CVR × bid, conversion funnel ESMM) ⑥ budget pacing. **Evaluation**: offline AUC + **calibration (ECE/logloss)** + online A/B (revenue, ROI, advertiser satisfaction, user-experience guardrails). Interview line: *"An ads CTR system ranks by eCPM = bid × pCTR in an auction, so pCTR must be calibrated (a true probability); good ranking ≠ accurate probability, and a high-AUC model that systematically overestimates wrecks the auction and billing — calibrate with Platt/isotonic and watch ECE not just AUC; challenges are billion-scale, extreme class imbalance, selection bias from observing only shown ads (needs exploration), freshness; sparse cross features use DeepFM/DCN, multi-objective CTR×CVR uses ESMM."*


In [ ]:

# ============================================================
# 核心:排序好 ≠ 概率准, 校准如何修正 / core: good ranking ≠ accurate probability; how calibration fixes it
# 中文:造一个点击数据(低基础点击率, 像广告)。用一个"排序很好但系统性高估概率"的模型, 看它 AUC 高但校准差,
#      再用保序回归校准——AUC 不变, 但概率变准。这正是广告必须做校准的原因。
# English: build click data (low base CTR, like ads). Use a model that "ranks well but systematically overestimates
#      probabilities"; its AUC is high but calibration is poor. Then calibrate with isotonic regression — AUC unchanged, probabilities accurate.
# ============================================================
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.isotonic import IsotonicRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
np.random.seed(0)
X=np.random.randn(30000,10); w=np.random.randn(10)
y=(np.random.rand(30000) < 1/(1+np.exp(-(X@w-2.0)))).astype(int)                 # 低基础点击率 / low base CTR
Xtr,Xrest,ytr,yrest=train_test_split(X,y,test_size=0.5,random_state=0)
Xcal,Xte,ycal,yte=train_test_split(Xrest,yrest,test_size=0.5,random_state=1)     # 校准集 + 测试集 / calibration + test
clf=LogisticRegression(max_iter=1000).fit(Xtr,ytr)
logit=lambda x: np.log(x/(1-x))
# 一个"排序好但系统性高估"的分数(单调畸变 → AUC 不变, 校准变差)/ a well-ranking but overconfident score
distort=lambda proba: 1/(1+np.exp(-(logit(proba)*1.8+1.2)))
mis_te =distort(clf.predict_proba(Xte)[:,1])                                      # 测试集的畸变(未校准)分数 / miscalibrated test scores
mis_cal=distort(clf.predict_proba(Xcal)[:,1])                                     # 校准集的畸变分数 / on calibration set
# 保序回归校准:把畸变分数映射回真实频率(在校准集上学映射)/ isotonic calibration: map scores → true frequency
iso=IsotonicRegression(out_of_bounds="clip").fit(mis_cal, ycal)
cal_te=iso.predict(mis_te)
def ece(p,y,bins=10):                                                             # 期望校准误差 / Expected Calibration Error
    e=0
    for i in range(bins):
        m=(p>=i/bins)&(p<(i+1)/bins)
        if m.sum()>0: e+=abs(p[m].mean()-y[m].mean())*m.sum()
    return e/len(p)
print(f"平均点击率 base CTR: {y.mean():.3f}")
print(f"AUC(排序能力):     校准前 {roc_auc_score(yte,mis_te):.3f}   校准后 {roc_auc_score(yte,cal_te):.3f}  ← 几乎不变(排序能力相同)")
print(f"ECE(校准误差):     校准前 {ece(mis_te,yte):.4f}   校准后 {ece(cal_te,yte):.4f}  ← 降 {ece(mis_te,yte)/max(ece(cal_te,yte),1e-9):.1f}x")
print("\n关键:同一个模型 AUC 一样(排序一样准), 但校准前概率系统性偏高(ECE 大), 校准后概率变准。")
print("eCPM=bid×pCTR: 未校准的 pCTR 让出价虚高→竞价排错、广告主超支/平台亏损。广告必须校准(推荐可不必)。")


In [ ]:

# ============================================================
# 可视化:可靠性曲线(校准前后)/ reliability diagram (before/after calibration)
# ============================================================
import matplotlib.pyplot as plt
def reliability(p,y,bins=10):
    xs,ys=[],[]
    for i in range(bins):
        m=(p>=i/bins)&(p<(i+1)/bins)
        if m.sum()>20: xs.append(p[m].mean()); ys.append(y[m].mean())
    return xs,ys
fig,ax=plt.subplots(1,2,figsize=(14,5))
# ① 可靠性曲线 / reliability curve
ax[0].plot([0,1],[0,1],"k--",alpha=0.5,label="完美校准(预测=实际)")
xr,yr=reliability(mis_te,yte); ax[0].plot(xr,yr,"o-",color="#C44E52",label=f"校准前(系统性高估, ECE={ece(mis_te,yte):.3f})")
xc,yc=reliability(cal_te,yte); ax[0].plot(xc,yc,"o-",color="#55A868",label=f"校准后(贴合对角线, ECE={ece(cal_te,yte):.3f})")
ax[0].set_xlabel("预测点击率 pCTR"); ax[0].set_ylabel("实际点击率"); ax[0].set_title("可靠性曲线:校准让预测概率贴合真实频率"); ax[0].legend(fontsize=8)
# ② 竞价影响示意:未校准 pCTR 导致出价错 / auction impact
ax[1].axis("off"); ax[1].set_title("为什么广告必须校准:eCPM=bid×pCTR",fontsize=12,weight="bold")
ax[1].text(0.5,0.8,"广告A: bid=\$2, 真实CTR=2%  → eCPM 应=\$0.04",ha="center",fontsize=9,transform=ax[1].transAxes)
ax[1].text(0.5,0.68,"广告B: bid=\$1, 真实CTR=5%  → eCPM 应=\$0.05 (B 应胜)",ha="center",fontsize=9,transform=ax[1].transAxes)
ax[1].text(0.5,0.48,"若模型把 A 的 pCTR 高估到 4%:",ha="center",fontsize=9,color="#C44E52",transform=ax[1].transAxes)
ax[1].text(0.5,0.36,"eCPM_A=\$2×4%=\$0.08 > eCPM_B → A 错误地胜出",ha="center",fontsize=9,color="#C44E52",transform=ax[1].transAxes)
ax[1].text(0.5,0.16,"排错广告 + 广告主为虚高预期点击超付 + 平台长期亏损\n→ 校准让 pCTR 可信, 竞价才正确",ha="center",fontsize=9,style="italic",transform=ax[1].transAxes)
plt.tight_layout(); plt.savefig("/tmp/sd04_viz.png",dpi=80); plt.show()
print("左:校准把系统性高估的曲线拉回对角线(预测=实际); 右:未校准 pCTR 让竞价排错广告、计费出错")


**中文**:诚实解读:
**English**: Honest takeaways:

**中文**:
1. **广告 CTR 系统的灵魂是"概率要准",不只是"排序要对"——这是它区别于所有其他排序系统的本质**:推荐、搜索、feed 都只要**排序对**(把好的排前面),预测分是 0.3 还是 0.6 无所谓。但广告不行:eCPM = bid × pCTR,pCTR 的**绝对数值**直接乘进出价,决定竞价胜负和计费金额。我们的 demo 一针见血:一个模型可以有很高的 AUC(排序能力一流),却系统性地把点击率高估——**AUC 完全没有告诉你这件事**(AUC 只看排序,不看概率的绝对准确性)。如果你在广告面试里只谈 AUC 不谈校准,是致命的知识盲区。校准(Platt/保序回归)在不改变排序的前提下,把预测概率拉回真实频率(我们把 ECE 降了 5 倍),让 eCPM 计算可信。**"排序好 ≠ 概率准,广告必须校准"是这道题的核心洞察。**
2. **广告系统是"极端规模 + 极端不平衡 + 反馈回路"的三重压力**:①**规模**:每秒百万次竞价、每天十亿次预测、毫秒级延迟——这决定了模型和特征工程要为极致吞吐优化(特征哈希、稀疏 embedding、模型量化),也是为什么广告是大规模稀疏 ML(FM/FFM/DeepFM/DCN)的发源地。②**极端不平衡**:点击率常常低于 1%,正样本极稀——负采样、类别权重、以及 log loss/校准(而非 accuracy)是标配。③**反馈回路/选择偏差(最深的坑)**:你只能观测到**展示过的广告**的点击,没展示的广告根本没有标签——这意味着训练数据是被你自己的系统筛选过的(selection bias),模型会陷入"只推它以为好的、永远不试新的"的死循环。破解要靠**探索(exploration)** 和无偏估计(接 19 因果、20.12)。这三重压力叠加,让广告 CTR 成为工业界最硬核的 ML 系统之一。
3. **诚实的复杂性与责任**:①**校准会漂移**:数据分布随时间变(节假日、新广告主、市场变化),昨天校准好的模型今天可能又偏了——所以要**持续监控校准(ECE)并定期重新校准**(接 22.10 漂移监控)。②**多目标与转化漏斗**:真实广告不只优化点击,还要转化(CTR × CVR × bid),而转化数据更稀疏、更延迟(用户点了广告可能几天后才购买),需要 ESMM 这类处理转化漏斗的模型(接 20.11)。③**预算平滑(pacing)**:广告主的日预算要平滑花完,不能上午就烧光——这是额外的约束优化。④**利益冲突的伦理**:广告系统同时服务三方——用户(想要好体验、不被骚扰)、广告主(想要转化、不想被骗超付)、平台(想要收入)。纯粹最大化平台收入会损害用户体验(广告过多过烦)和广告主信任(计费不透明)。**负责任的广告系统要平衡三方**,用户体验护栏(广告密度、相关性、不推有害广告)是必须的。**结论:设计广告 CTR 系统的核心是竞价排序 eCPM=bid×pCTR, 决定性要求是校准(pCTR 必须是真实概率, 排序好≠概率准, 用 Platt/保序回归, 看 ECE 而非只看 AUC); 三重压力是极端规模、极端类别不平衡、只观测展示广告的选择偏差(需探索); 还要处理校准漂移、转化漏斗多目标、预算平滑, 并在用户/广告主/平台三方利益间做负责任的平衡——这是工业界最硬核的 ML 系统之一。**

**English**:
1. **The soul of an ads-CTR system is "accurate probabilities," not just "correct ranking" — this is what fundamentally distinguishes it from all other ranking systems**: recommendation, search, and feed only need **correct ranking** (good things first); whether the predicted score is 0.3 or 0.6 doesn't matter. But ads can't: eCPM = bid × pCTR, and pCTR's **absolute value** multiplies directly into the bid, deciding auction winners and billing amounts. Our demo cuts to it: a model can have very high AUC (top-tier ranking) yet systematically overestimate click rates — **AUC tells you nothing about this** (AUC only sees ranking, not probability's absolute accuracy). If you discuss only AUC without calibration in an ads interview, it's a fatal blind spot. Calibration (Platt/isotonic) pulls predicted probabilities back to true frequencies without changing ranking (we cut ECE 5x), making the eCPM computation trustworthy. **"Good ranking ≠ accurate probability, ads must be calibrated" is this question's core insight.**
2. **Ad systems are under the triple pressure of "extreme scale + extreme imbalance + feedback loops"**: ① **Scale**: millions of auctions per second, billions of predictions per day, millisecond latency — dictating that models and feature engineering optimize for extreme throughput (feature hashing, sparse embeddings, model quantization), and why ads birthed large-scale sparse ML (FM/FFM/DeepFM/DCN). ② **Extreme imbalance**: click rates often below 1%, positives extremely rare — negative sampling, class weights, and log loss/calibration (not accuracy) are standard. ③ **Feedback loops/selection bias (the deepest trap)**: you can only observe clicks on **shown ads**; unshown ads have no labels at all — meaning training data is filtered by your own system (selection bias), and the model falls into "only serve what it thinks is good, never trying new," a dead loop. Breaking it requires **exploration** and unbiased estimation (per 19 causal, 20.12). These three pressures combined make ads-CTR one of industry's most hardcore ML systems.
3. **Honest complexity and responsibility**: ① **Calibration drifts**: data distribution changes over time (holidays, new advertisers, market shifts), so a model calibrated yesterday may be off today — so **continuously monitor calibration (ECE) and periodically recalibrate** (per 22.10 drift monitoring). ② **Multi-objective and the conversion funnel**: real ads optimize not just clicks but conversions (CTR × CVR × bid), and conversion data is sparser and more delayed (a user clicks an ad but may buy days later), needing ESMM-like models for the conversion funnel (per 20.11). ③ **Budget pacing**: an advertiser's daily budget must be spent smoothly, not burned by morning — an extra constrained optimization. ④ **The ethics of conflicting interests**: the ad system serves three parties simultaneously — users (want good experience, no harassment), advertisers (want conversions, no fraudulent overspend), the platform (wants revenue). Purely maximizing platform revenue harms user experience (too many annoying ads) and advertiser trust (opaque billing). **A responsible ad system balances all three**, and user-experience guardrails (ad density, relevance, no harmful ads) are mandatory. **Conclusion: designing an ads-CTR system centers on auction ranking eCPM = bid × pCTR, with the decisive requirement of calibration (pCTR must be a true probability; good ranking ≠ accurate probability; use Platt/isotonic, watch ECE not just AUC); the triple pressure is extreme scale, extreme class imbalance, and selection bias from observing only shown ads (needs exploration); also handle calibration drift, the conversion-funnel multi-objective, budget pacing, and responsibly balance user/advertiser/platform interests — one of industry's most hardcore ML systems.**

> 💼 **实战视角 / Practical angle**
> **中文**:广告 CTR 落地:①**模型**:大规模稀疏特征(用户×广告×上下文交叉), DeepFM/DCN(记忆+泛化)、embedding、特征哈希;②**必做校准**:Platt/保序回归, 监控 **ECE 和 log loss**(不只 AUC), 因为 eCPM=bid×pCTR 依赖概率绝对值; 校准会漂移要定期重校;③**类别不平衡**:负采样(记得校正采样偏差)、用 log loss;④**选择偏差/探索**:只观测展示广告的点击→用 ε-greedy/bandit/无偏估计给新广告机会;⑤**多目标**:CTR×CVR 用 ESMM(转化漏斗、延迟转化);⑥**预算平滑** pacing;⑦**在线学习**追时效;⑧**评估**:离线 AUC+校准+log loss, 在线 A/B 看收入/ROI/用户体验护栏。**答题**:强调 eCPM 排序→必须校准(排序好≠概率准)、极端规模不平衡、选择偏差需探索、三方利益平衡。面试金句:*"广告按 eCPM=bid×pCTR 竞价, 所以 pCTR 必须校准(真实概率)——高AUC模型系统性偏差会毁掉竞价计费, 用保序回归/Platt 校准、看 ECE 不只 AUC; 挑战是十亿级规模、CTR<1% 极不平衡、只观测展示广告的选择偏差(需探索)、校准漂移、转化漏斗多目标(ESMM)、预算平滑; 还要平衡用户/广告主/平台三方。"*
> **English**: Ads-CTR in practice: ① **models**: large-scale sparse features (user × ad × context crosses), DeepFM/DCN (memorization + generalization), embeddings, feature hashing; ② **mandatory calibration**: Platt/isotonic, monitor **ECE and log loss** (not just AUC), because eCPM = bid × pCTR depends on absolute probability; calibration drifts so recalibrate periodically; ③ **class imbalance**: negative sampling (remember to correct the sampling bias), use log loss; ④ **selection bias/exploration**: observing only shown-ad clicks → use ε-greedy/bandit/unbiased estimation to give new ads a chance; ⑤ **multi-objective**: CTR × CVR via ESMM (conversion funnel, delayed conversions); ⑥ **budget pacing**; ⑦ **online learning** for freshness; ⑧ **evaluation**: offline AUC + calibration + log loss, online A/B for revenue/ROI/user-experience guardrails. **Answering**: emphasize eCPM ranking → mandatory calibration (good ranking ≠ accurate probability), extreme scale/imbalance, selection bias needing exploration, three-party balance. Interview line: *"Ads bid via eCPM = bid × pCTR, so pCTR must be calibrated (a true probability) — a high-AUC model with systematic bias wrecks the auction and billing; calibrate with isotonic/Platt, watch ECE not just AUC; challenges are billion-scale, CTR<1% extreme imbalance, selection bias from observing only shown ads (needs exploration), calibration drift, conversion-funnel multi-objective (ESMM), budget pacing; also balance user/advertiser/platform interests."*

---
### 小结 / Summary
- **中文**:广告按 eCPM=bid×pCTR 竞价排序(GSP 二价); 决定性要求=校准(pCTR 必须是真实概率, 绝对值进出价)。
- **English**: Ads rank by auction eCPM = bid × pCTR (GSP second-price); decisive requirement = calibration (pCTR must be a true probability, its absolute value enters the bid).
- **中文**:排序好(高 AUC)≠ 概率准; 用 Platt/保序回归校准, 看 ECE/log loss 而非只看 AUC(demo: AUC 不变 ECE 降 5x)。
- **English**: Good ranking (high AUC) ≠ accurate probability; calibrate with Platt/isotonic, watch ECE/log loss not just AUC (demo: AUC unchanged, ECE cut 5x).
- **中文**:三重压力=极端规模+极端类别不平衡+选择偏差(只观测展示广告, 需探索); 还有校准漂移、转化漏斗、预算平滑、三方利益平衡。
- **English**: Triple pressure = extreme scale + extreme class imbalance + selection bias (observing only shown ads, needs exploration); plus calibration drift, conversion funnel, budget pacing, three-party balance.
